In [0]:
# Install Snowflake connector for Python
%pip install snowflake-connector-python

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.9/2.9 MB 15.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.7/4.7 MB 33.3 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.3
    Not uninstalling requests at /databricks/python3/lib/python3.12/site-packages, outside environment /local_disk0/.ephemeral_nfs/envs/pythonEnv-4e388a4d-f215-441e-96e4-32b2c1b7c22c
    Can't uninstall 'requests'. No files were found to uninstall.
  Attempting uninstall: cffi
    Found existing installation: cffi 1.17.1
    Not uninstalling cffi at /databricks/python3/lib/python3.12/site-packages, outside environment /local_disk0/.ephemeral_nfs/envs/pythonEnv-4e388a4d-f215-441e-96e4-32b2c1b7c22c
    Can't uninstall 'cffi'. No files were found to uninstall.
  Attempting uninstall: cryptography
    Found existing installation: cryptography 44.0.1
    Not uninstalling cryptography at /databricks/python3/lib/python3.12/site-packages, outside environment /lo

In [0]:
# Restart Python kernel to use new packages
dbutils.library.restartPython()

In [0]:
# Import Snowflake connector
import snowflake.connector

print(f"Snowflake connector version: {snowflake.connector.__version__}")

Snowflake connector version: 4.5.0


In [0]:
# Configure Snowflake connection parameters
# Replace these with your actual Snowflake credentials
# For production, use dbutils.secrets.get() to securely retrieve credentials

snowflake_config = {
    'user': 'jeevan066',
    'password': '8105270660Jj@@',
    'account': 'MRQRVHW-ET45819',  
    'warehouse': 'INSURANCE_WH',
    'database': 'INSURANCE_WH',
    'schema': 'GOLD'
}

# Example using secrets (recommended for production):
# snowflake_config = {
#     'user': dbutils.secrets.get(scope='snowflake', key='username'),
#     'password': dbutils.secrets.get(scope='snowflake', key='password'),
#     'account': dbutils.secrets.get(scope='snowflake', key='account'),
#     'warehouse': '<your_warehouse>',
#     'database': '<your_database>',
#     'schema': '<your_schema>'
# }

print("Snowflake configuration ready (credentials masked)")

Snowflake configuration ready (credentials masked)


In [0]:
# Establish connection to Snowflake
try:
    conn = snowflake.connector.connect(**snowflake_config)
    
    # Test the connection
    cursor = conn.cursor()
    cursor.execute("SELECT CURRENT_VERSION()")
    version = cursor.fetchone()
    
    print(f"Successfully connected to Snowflake!")
    print(f"Snowflake version: {version[0]}")
    print(f"Connected to warehouse: {snowflake_config['warehouse']}")
    print(f"Database: {snowflake_config['database']}, Schema: {snowflake_config['schema']}")
    
    cursor.close()
    
except Exception as e:
    print(f"Error connecting to Snowflake: {str(e)}")
    print("\nPlease verify your credentials and connection parameters.")

Successfully connected to Snowflake!
Snowflake version: 10.18.101
Connected to warehouse: INSURANCE_WH
Database: INSURANCE_WH, Schema: GOLD


In [0]:
# re-read all gold delta tables after restart

df_claims_summary=spark.read \
    .format("delta")\
    .table("workspace.insurance_claims.gold_claims_summary")
        
df_fraud_analysis=spark.read \
    .format("delta")\
    .table("workspace.insurance_claims.gold_fraud_analysis")
        
df_customer_360=spark.read \
    .format("delta")\
    .table("workspace.insurance_claims.gold_customer_360")

print("Claims Summar", df_claims_summary.count())
print("Fraud Analysis", df_fraud_analysis.count())
print("Customer 360", df_customer_360.count())

Claims Summar 992
Fraud Analysis 862
Customer 360 1000


In [0]:
import pandas as pd
from snowflake.connector.pandas_tools import write_pandas

#function to load to snowflake
def load_to_snowflake(df_spark, table_name):
    # convert spark to pandas
    df_pandas=df_spark.toPandas()

    # uppercase column names - Snowflake rewuirement
    df_pandas.columns=[c.upper() for c in df_pandas.columns]
    
    # Create a fresh pandas DataFrame to avoid serialization issues
    df_pandas = pd.DataFrame(df_pandas.values, columns=df_pandas.columns)

    # truncate first
    cur=conn.cursor()
    cur.execute("USE DATABASE INSURANCE_DW")
    cur.execute(f"TRUNCATE TABLE GOLD.{table_name}")
    

    # load to snowflake
    success, nchunks, nrows, _ = write_pandas(conn=conn, df=df_pandas, table_name=table_name, schema="GOLD", database="INSURANCE_DW")
    print(f"{table_name}: {nrows} rows loaded")
    cur.close()
  # load all 3 gold tables
load_to_snowflake(df_claims_summary,"CLAIMS_SUMMARY")
load_to_snowflake(df_fraud_analysis, "FRAUD_ANALYSIS")
load_to_snowflake(df_customer_360,"CUSTOMER_360")

print("\n All Gold data loaded to Snowflake !")

CLAIMS_SUMMARY: 992 rows loaded
FRAUD_ANALYSIS: 862 rows loaded
CUSTOMER_360: 1000 rows loaded

 All Gold data loaded to Snowflake !
